# 1. Установка Goggle Drive для успешной работы в на облачном диске

In [1]:
from google.colab import drive
drive.mount('/gdrive')
# the project's folder
%cd /gdrive/'My Drive'

Mounted at /gdrive
/gdrive/My Drive


# 2. Клонирование репозитория YOLOv5, установка зависимостей, проверка готовности.

In [2]:
!git clone https://github.com/ultralytics/yolov5  # clone

Cloning into 'yolov5'...
remote: Enumerating objects: 17270, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 17270 (delta 0), reused 0 (delta 0), pack-reused 17269 (from 2)
Receiving objects: 100% (17270/17270), 16.10 MiB | 10.04 MiB/s, done.
Resolving deltas: 100% (11855/11855), done.
Updating files: 100% (146/146), done.


In [3]:
%cd yolov5
%pip install -qr requirements.txt comet_ml  # install
import torch
import utils
display = utils.notebook_init()  # checks

YOLOv5 🚀 v7.0-398-g5cdad892 Python-3.11.11 torch-2.5.1+cu124 CUDA:0 (Tesla T4, 15095MiB)


Setup complete ✅ (2 CPUs, 12.7 GB RAM, 40.7/112.6 GB disk)


# 3. Детектирование объектов

`detect.py` запускает YOLOv5 детектирование объектов, файл модели загружается из [latest YOLOv5 release](https://github.com/ultralytics/yolov5/releases). Пример различных вариантов выполнения детекции:

```shell
python detect.py --source 0  # webcam
                          img.jpg  # image
                          vid.mp4  # video
                          screen  # screenshot
                          path/  # directory
                         'path/*.jpg'  # glob
                         'https://youtu.be/LNwODJXcvt4'  # YouTube
                         'rtsp://example.com/media.mp4'  # RTSP, RTMP, HTTP stream
```

In [8]:
!python detect.py --weights yolov5n.pt --img 640 --conf 0.25 --source /gdrive/MyDrive/img_detect --project /gdrive/MyDrive/detected --save-txt --save-conf


detect: weights=['yolov5n.pt'], source=/gdrive/MyDrive/img_detect, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=True, save_format=0, save_csv=False, save_conf=True, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=/gdrive/MyDrive/detected, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-398-g5cdad892 Python-3.11.11 torch-2.5.1+cu124 CUDA:0 (Tesla T4, 15095MiB)

100% 3.87M/3.87M [00:00<00:00, 98.0MB/s]

Fusing layers... 
YOLOv5n summary: 213 layers, 1867405 parameters, 0 gradients, 4.5 GFLOPs
image 1/20 /gdrive/MyDrive/img_detect/close-up-cleaning-sex-toys_23-2149151824.jpg_1_11zon.jpg: 448x640 1 cup, 1 chair, 1 clock, 43.3ms
image 2/20 /gdrive/MyDrive/img_detect/composition-with-summer-objects_23-2147647077.jpg_2_11zon.jpg: 640x448 5 oranges, 1 scissors, 46.3m

## После выполнения кода на гугл-диске создастся папка «detected», с подпапками вида exp. В папке exp будут находится детектированные YOLO изображения.

## Следующий код сохранит результаты детектирования с тектовыми файлами с метками объектов.
Вид метки:
class X Y W H conf
где class - класс объекта
 X Y - координаты ограничивающего прямоугольника
 W H  - высота и ширина прямоугольника
 conf - уверенность детекции объета

In [10]:
!python detect.py --weights yolov5n.pt --img 640 --conf 0.25 --source /gdrive/MyDrive/img_detect --project /gdrive/MyDrive/detected --save-txt --save-conf


detect: weights=['yolov5n.pt'], source=/gdrive/MyDrive/img_detect, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=True, save_format=0, save_csv=False, save_conf=True, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=/gdrive/MyDrive/detected, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-398-g5cdad892 Python-3.11.11 torch-2.5.1+cu124 CUDA:0 (Tesla T4, 15095MiB)

Fusing layers... 
YOLOv5n summary: 213 layers, 1867405 parameters, 0 gradients, 4.5 GFLOPs
image 1/20 /gdrive/MyDrive/img_detect/close-up-cleaning-sex-toys_23-2149151824.jpg_1_11zon.jpg: 448x640 1 cup, 1 chair, 1 clock, 27.7ms
image 2/20 /gdrive/MyDrive/img_detect/composition-with-summer-objects_23-2147647077.jpg_2_11zon.jpg: 640x448 5 oranges, 1 scissors, 28.8ms
image 3/20 /gdrive/MyDrive/img_detect/cr

In [11]:
import os
import shutil
import random

def split_dataset(source_path, dest_path, train_ratio=0.8, seed=42):
    """
    Разбивает датасет из source_path, где ожидается наличие папок 'images' и 'labels',
    и копирует файлы в новую структуру dest_path:

    dest_path/
        images/
            train/
            valid/
        labels/
            train/
            valid/

    Файлы распределяются в соотношении train_ratio (тренировочная выборка) и (1 - train_ratio) (валидация).

    Аргументы:
        source_path (str): путь к исходному датасету (например, "/gdrive/MyDrive/Apples_detect/LabelImg")
        dest_path (str): путь для сохранения распределённого датасета (например, "/gdrive/MyDrive/dataset/apples")
        train_ratio (float): доля файлов для тренировочного набора (по умолчанию 0.8)
        seed (int): зерно для генератора случайных чисел (для воспроизводимости)
    """
    random.seed(seed)

    # Исходные папки
    src_images_dir = os.path.join(source_path, "images")
    src_labels_dir = os.path.join(source_path, "labels")

    # Целевая структура
    dest_images_train = os.path.join(dest_path, "images", "train")
    dest_images_valid = os.path.join(dest_path, "images", "valid")
    dest_labels_train = os.path.join(dest_path, "labels", "train")
    dest_labels_valid = os.path.join(dest_path, "labels", "valid")

    # Создание целевых директорий, если их нет
    os.makedirs(dest_images_train, exist_ok=True)
    os.makedirs(dest_images_valid, exist_ok=True)
    os.makedirs(dest_labels_train, exist_ok=True)
    os.makedirs(dest_labels_valid, exist_ok=True)

    # Получаем список изображений (поддерживаются форматы jpg, jpeg, png)
    all_images = [f for f in os.listdir(src_images_dir)
                  if os.path.isfile(os.path.join(src_images_dir, f)) and
                  f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    # Перемешиваем файлы для случайного распределения
    random.shuffle(all_images)

    num_train = int(len(all_images) * train_ratio)
    train_images = all_images[:num_train]
    valid_images = all_images[num_train:]

    # Копирование файлов для тренировочного набора
    for fname in train_images:
        # Копирование изображения
        src_img = os.path.join(src_images_dir, fname)
        dst_img = os.path.join(dest_images_train, fname)
        shutil.copy(src_img, dst_img)

        # Копирование соответствующего файла разметки (если существует)
        label_fname = os.path.splitext(fname)[0] + ".txt"
        src_label = os.path.join(src_labels_dir, label_fname)
        if os.path.exists(src_label):
            dst_label = os.path.join(dest_labels_train, label_fname)
            shutil.copy(src_label, dst_label)

    # Копирование файлов для валидационного набора
    for fname in valid_images:
        src_img = os.path.join(src_images_dir, fname)
        dst_img = os.path.join(dest_images_valid, fname)
        shutil.copy(src_img, dst_img)

        label_fname = os.path.splitext(fname)[0] + ".txt"
        src_label = os.path.join(src_labels_dir, label_fname)
        if os.path.exists(src_label):
            dst_label = os.path.join(dest_labels_valid, label_fname)
            shutil.copy(src_label, dst_label)

    print(f"Разбиение завершено: {len(train_images)} изображений для тренировки, {len(valid_images)} изображений для валидации.")

if __name__ == "__main__":
    # Путь к исходному датасету с папками images и labels
    source_dataset_path = "/gdrive/MyDrive/Apples_detect/LabelIMG"
    # Путь к новому датасету с требуемой структурой
    destination_dataset_path = "/gdrive/MyDrive/dataset/apples"

    split_dataset(source_dataset_path, destination_dataset_path)

Разбиение завершено: 94 изображений для тренировки, 24 изображений для валидации.


## 4. Обучение модели YOLOv5

In [13]:
# %% [code]
!cp /gdrive/MyDrive/apples.yaml /gdrive/MyDrive/yolov5/data

Подключение tensorboard

In [14]:
%reload_ext tensorboard
# %tensorboard --logdir logs/fit

Датасет для обучения должен располагаться в папке dataset/apples в корневой папке гугл-диска. Структура папок следующая:

dataset/apples

              /images
                    /train
                    /valid
              /labels
                    /train
                    /valid


## Просмотр экспериментов с помощью tensorboard

### Тренируем данные на 10, 20 и 30 эпохах

In [20]:
!python train.py --img 640 --batch 16 --epochs 10 --data /gdrive/MyDrive/apples.yaml --weights yolov5n.pt --project /gdrive/MyDrive/yolov5/runs/train --name exp10
!python train.py --img 640 --batch 16 --epochs 20 --data /gdrive/MyDrive/apples.yaml --weights yolov5n.pt --project /gdrive/MyDrive/yolov5/runs/train --name exp20
!python train.py --img 640 --batch 16 --epochs 30 --data /gdrive/MyDrive/apples.yaml --weights yolov5n.pt --project /gdrive/MyDrive/yolov5/runs/train --name exp30

2025-03-13 13:36:44.964651: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1741873004.985836   10630 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1741873004.992280   10630 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
train: weights=yolov5n.pt, cfg=, data=/gdrive/MyDrive/apples.yaml, hyp=data/hyps/hyp.scratch-low.yaml, epochs=10, batch_size=16, imgsz=640, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None, evolve_population=data/hyps, resume_evolve=None, bucket=, cache=None, image_weights=False, device=, multi_scale=False, single_cls=False, optimizer=SGD, sync_bn=False, workers=8, project=/gdrive/My

In [27]:
# Если по пути /gdrive/MyDrive/yolov5/weights находится файл, удаляем его
!if [ -f /gdrive/MyDrive/yolov5/weights ]; then rm /gdrive/MyDrive/yolov5/weights; fi

# Создаем директорию для весов
!mkdir -p /gdrive/MyDrive/yolov5/weights

# Копируем лучшие веса из результатов обучения в созданную директорию
!cp /gdrive/MyDrive/yolov5/runs/train/exp10/weights/best.pt /gdrive/MyDrive/yolov5/weights/best_10.pt
!cp /gdrive/MyDrive/yolov5/runs/train/exp20/weights/best.pt /gdrive/MyDrive/yolov5/weights/best_20.pt
!cp /gdrive/MyDrive/yolov5/runs/train/exp30/weights/best.pt /gdrive/MyDrive/yolov5/weights/best_30.pt


In [28]:
!python detect.py --weights weights/best_20.pt --img 640 --conf 0.25 --source ../img_detect --project ../detected --save-txt --save-conf

detect: weights=['weights/best_20.pt'], source=../img_detect, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=True, save_format=0, save_csv=False, save_conf=True, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=../detected, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-398-g5cdad892 Python-3.11.11 torch-2.5.1+cu124 CUDA:0 (Tesla T4, 15095MiB)

Fusing layers... 
Model summary: 157 layers, 1764577 parameters, 0 gradients, 4.1 GFLOPs
image 1/20 /gdrive/MyDrive/img_detect/close-up-cleaning-sex-toys_23-2149151824.jpg_1_11zon.jpg: 448x640 9 tennis_balls, 1 red_apple, 29.5ms
image 2/20 /gdrive/MyDrive/img_detect/composition-with-summer-objects_23-2147647077.jpg_2_11zon.jpg: 640x448 20 tennis_balls, 1 yellow_apple, 31.8ms
image 3/20 /gdrive/MyDrive/img_detect/creative

# 5. Просмотр результатов обучения модели

In [31]:
import os
import pandas as pd

def load_boxes(file_path, is_prediction=False):
    """
    Загружает разметку из файла.
    Для ground truth формат: class x_center y_center width height
    Для предсказаний формат: class x_center y_center width height confidence
    """
    boxes = []
    if not os.path.exists(file_path):
        return boxes
    with open(file_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if is_prediction:
                if len(parts) == 6:
                    cls, x, y, w, h, conf = parts
                    boxes.append({
                        'class': int(cls),
                        'bbox': [float(x), float(y), float(w), float(h)],
                        'conf': float(conf)
                    })
            else:
                if len(parts) == 5:
                    cls, x, y, w, h = parts
                    boxes.append({
                        'class': int(cls),
                        'bbox': [float(x), float(y), float(w), float(h)]
                    })
    return boxes

def iou(box1, box2):
    """
    Вычисляет Intersection over Union (IoU) для двух прямоугольников,
    заданных в формате YOLO: [x_center, y_center, width, height].
    """
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2
    xmin1 = x1 - w1 / 2
    ymin1 = y1 - h1 / 2
    xmax1 = x1 + w1 / 2
    ymax1 = y1 + h1 / 2
    xmin2 = x2 - w2 / 2
    ymin2 = y2 - h2 / 2
    xmax2 = x2 + w2 / 2
    ymax2 = y2 + h2 / 2

    inter_xmin = max(xmin1, xmin2)
    inter_ymin = max(ymin1, ymin2)
    inter_xmax = min(xmax1, xmax2)
    inter_ymax = min(ymax1, ymax2)

    inter_width = max(0, inter_xmax - inter_xmin)
    inter_height = max(0, inter_ymax - inter_ymin)
    inter_area = inter_width * inter_height
    area1 = w1 * h1
    area2 = w2 * h2
    union_area = area1 + area2 - inter_area
    if union_area == 0:
        return 0
    return inter_area / union_area

def compute_detection_metrics(gt_dir, pred_dir, iou_threshold=0.5):
    """
    Сравнивает ground truth и предсказания для каждого файла и вычисляет метрики:
    Accuracy, Error Rate, Precision, Recall.
    """
    total_TP = 0
    total_FP = 0
    total_FN = 0

    # Список файлов ground truth (ожидается расширение .txt)
    gt_files = [f for f in os.listdir(gt_dir) if f.endswith('.txt')]

    for file in gt_files:
        gt_path = os.path.join(gt_dir, file)
        pred_path = os.path.join(pred_dir, file)

        gt_boxes = load_boxes(gt_path, is_prediction=False)
        pred_boxes = load_boxes(pred_path, is_prediction=True)

        matched_gt = []
        TP = 0
        FP = 0

        for pred in pred_boxes:
            best_iou = 0
            best_gt_idx = -1
            for idx, gt in enumerate(gt_boxes):
                if gt['class'] == pred['class'] and idx not in matched_gt:
                    current_iou = iou(gt['bbox'], pred['bbox'])
                    if current_iou > best_iou:
                        best_iou = current_iou
                        best_gt_idx = idx
            if best_iou >= iou_threshold and best_gt_idx != -1:
                TP += 1
                matched_gt.append(best_gt_idx)
            else:
                FP += 1

        FN = len(gt_boxes) - len(matched_gt)
        total_TP += TP
        total_FP += FP
        total_FN += FN

    precision = total_TP / (total_TP + total_FP) if (total_TP + total_FP) > 0 else 0
    recall = total_TP / (total_TP + total_FN) if (total_TP + total_FN) > 0 else 0
    accuracy = total_TP / (total_TP + total_FP + total_FN) if (total_TP + total_FP + total_FN) > 0 else 0
    error_rate = 1 - accuracy

    return accuracy, error_rate, precision, recall

# Пути к ground truth и предсказаниям для каждой модели
gt_dir = "/gdrive/MyDrive/dataset/apples/labels/valid"
pred_dir_10 = "/gdrive/MyDrive/detected/exp10/exp3/labels"
pred_dir_20 = "/gdrive/MyDrive/detected/exp20/exp3/labels"
pred_dir_30 = "/gdrive/MyDrive/detected/exp30/exp3/labels"

# Вычисляем метрики для моделей, обученных на 10, 20 и 30 эпох
metrics_10 = compute_detection_metrics(gt_dir, pred_dir_10)
metrics_20 = compute_detection_metrics(gt_dir, pred_dir_20)
metrics_30 = compute_detection_metrics(gt_dir, pred_dir_30)

# Формируем таблицу с метриками
data = {
    "Модель на 10 эпох": [f"{metrics_10[0]:.2f}", f"{metrics_10[1]:.2f}", f"{metrics_10[2]:.2f}", f"{metrics_10[3]:.2f}"],
    "Модель на 20 эпох": [f"{metrics_20[0]:.2f}", f"{metrics_20[1]:.2f}", f"{metrics_20[2]:.2f}", f"{metrics_20[3]:.2f}"],
    "Модель на 30 эпох": [f"{metrics_30[0]:.2f}", f"{metrics_30[1]:.2f}", f"{metrics_30[2]:.2f}", f"{metrics_30[3]:.2f}"]
}
metrics_df = pd.DataFrame(data, index=["Accuracy", "Error Rate", "Precision", "Recall"])

print("Сравнение метрик моделей:")
print(metrics_df)


Сравнение метрик моделей:
            Модель на 10 эпох  Модель на 20 эпох  Модель на 30 эпох
Метрика                                                            
Accuracy                 0.65               0.79               0.77
Error Rate               0.35               0.21               0.23
Precision                0.60               0.76               0.72
Recall                   0.58               0.73               0.70
+------------+---------------------+---------------------+---------------------+
| Метрика    |   Модель на 10 эпох |   Модель на 20 эпох |   Модель на 30 эпох |
|------------+---------------------+---------------------+---------------------|
| Accuracy   |                0.65 |                0.79 |                0.77 |
| Error Rate |                0.35 |                0.21 |                0.23 |
| Precision  |                0.6  |                0.76 |                0.72 |
| Recall     |                0.58 |                0.73 |                0.7  |

## Локальные данные

Результаты обучения автоматически регистрируются в [Tensorboard](https://www.tensorflow.org/tensorboard) and [CSV](https://github.com/ultralytics/yolov5/pull/4148) , а также в папке `runs/train`, где создаются подпапки с результатами различных экспериментов в виде `runs/train/exp2`, `runs/train/exp3`, etc.

Этот каталог содержит статистику обучения и проверки, мозаики, метки, прогнозы и дополненные мозаики, а также показатели и графики, в том числе кривые точности-отзыва (PR) и матрицы ошибок.

<img alt="Local logging results" src="https://user-images.githubusercontent.com/26833433/183222430-e1abd1b7-782c-4cde-b04d-ad52926bf818.jpg" width="1280"/>
